# Project Task: Compositional Image Retrieval
This Notebook explores the task of compositional image retrieval, where the goal is to retrieve images based on a combination of visual and textual inputs. 

We present and evaluate different approaches to this task, comparing their performance with a baseline method.

## Environment Setup

This notebook is designed for Google Colab.

Before running, ensure:
- Google Drive is mounted.
- The dataset zip exists at `/content/drive/MyDrive/datasets/celeba.zip`.
- You run cells top-to-bottom at least once to initialize all variables.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Create the directory if it doesn't exist
!mkdir -p /content/datasets

In [ ]:
# This should take 1-2 minutes
# It unzips the dataset in the runtime's local SSD, so when
# you disconnect, it gets deleted
!unzip -q /content/drive/MyDrive/datasets/celeba.zip -d /content/datasets/

### Imports

In [ ]:
import torch
from pathlib import Path
import os
import json
from typing import Callable
from functools import partial
import matplotlib.pyplot as plt
import numpy as np

from transformers import CLIPModel, CLIPProcessor
from torchvision.datasets import CelebA
import torch.nn.functional as F

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
MODEL_NAME = "openai/clip-vit-base-patch32"

### Encoding utilities
Common CLIP encoding utilities for processing images and text, shared across the notebook. 
These wrap the CLIP model for encoding both images and text into a common embedding space, with a single source of truth for encoding logic.

In [ ]:
_model = None
_processor = None


def get_CLIP_model():
    global _model, _processor
    if _model is None:
        print("Loading CLIP model...")
        _model = CLIPModel.from_pretrained(MODEL_NAME).to(device)
        _model.eval()
    if _processor is None:
        _processor = CLIPProcessor.from_pretrained(MODEL_NAME)
    return _model, _processor


def _as_feature_tensor(out) -> torch.Tensor:
    """
    Normalize CLIPModel.get_text_features / get_image_features outputs into a Tensor.
    Different transformers versions return either a plain Tensor or a
    BaseModelOutputWithPooling-style object exposing .text_embeds / .image_embeds /
    .pooler_output.
    """
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("text_embeds", "image_embeds", "pooler_output"):
        v = getattr(out, attr, None)
        if v is not None:
            return v
    if isinstance(out, tuple) and len(out) > 0:
        return out[0]
    raise TypeError(f"Unexpected feature output type: {type(out)}")


@torch.no_grad()
def encode_texts(prompts: list[str], device) -> torch.Tensor:
    """Encode a batch of prompts in one call. Returns (P, D), L2-normalized per row, on `device`."""
    model, processor = get_CLIP_model()
    inputs = processor(text=prompts, return_tensors="pt", padding=True, truncation=True).to(device)
    embs = _as_feature_tensor(model.get_text_features(**inputs))
    return F.normalize(embs, p=2, dim=-1)


@torch.no_grad()
def encode_text(prompt: str, device) -> torch.Tensor:
    """Tokenize, encode, and L2-normalize a single text prompt. Returns shape (D,) on `device`."""
    return encode_texts([prompt], device).view(-1)


@torch.no_grad()
def encode_images(images, device, batch_size: int = 128, to_cpu: bool = True, log_every: int = 1000) -> torch.Tensor:
    """Encode an iterable of PIL images. Returns (N, D), L2-normalized per row.
    Streams in `batch_size` chunks so callers don't have to hand-roll batching.
    Each batch is moved to CPU before concatenation by default to keep GPU memory
    bounded when encoding large datasets. Pass `to_cpu=False` to keep the result
    on `device` (useful for single-image / small-batch calls).
    """
    model, processor = get_CLIP_model()
    images = list(images)
    n_total = len(images)

    feats: list[torch.Tensor] = []
    buf: list = []
    n_encoded = 0

    def _flush():
        nonlocal n_encoded
        inputs = processor(images=buf, return_tensors="pt").to(device)
        e = _as_feature_tensor(model.get_image_features(**inputs))
        e = F.normalize(e, p=2, dim=-1)
        feats.append(e.cpu() if to_cpu else e)
        n_encoded += len(buf)
        buf.clear()

    for img in images:
        buf.append(img)
        if len(buf) >= batch_size:
            _flush()
            if n_encoded % log_every < batch_size:
                pct = 100 * n_encoded / n_total
                print(f"Encoded {n_encoded:>{len(str(n_total))}}/{n_total} images ({pct:.1f}%)")

    if buf:
        _flush()

    print(f"Encoded {n_encoded}/{n_total} images (100.0%) — done.")
    return torch.cat(feats, dim=0) if feats else torch.empty(0)


@torch.no_grad()
def encode_image(image, device) -> torch.Tensor:
    """Encode and L2-normalize a single PIL image. Returns shape (D,) on `device`."""
    return encode_images([image], device, to_cpu=False).view(-1)


@torch.no_grad()
def get_encoded_dataset(
    dataset,
    device,
    cache_path: str,
    batch_size: int = 128,
    num_workers: int = 4,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Encode all images in `dataset` and return (features, labels).
      - features: (N, D) on `device`, L2-normalized per row.
      - labels:   (N, ...) on CPU, as produced by the dataset.

    If a cached file exists at `cache_path`, load and return it. Otherwise
    compute features and labels in a single DataLoader pass, cache as a dict
    {"features", "labels"}, and return. Legacy caches that hold a raw features
    tensor (no labels) are still readable: labels are re-stacked from `dataset`
    without re-encoding images, and the cache file is left untouched.
    """
    os.makedirs(os.path.dirname(cache_path), exist_ok=True)

    if os.path.exists(cache_path):
        print(f"Loading cached features from {cache_path}.")
        blob = torch.load(cache_path, map_location="cpu")
        if isinstance(blob, dict):
            features = blob["features"].to(device)
            labels   = blob["labels"]
        else:
            # Legacy cache: features only — re-stack labels without re-encoding.
            features = blob.to(device)
            labels   = torch.stack([lbl for _img, lbl in dataset], dim=0)
        print(f"Loaded from cache. features: {tuple(features.shape)}, labels: {tuple(labels.shape)}")
        return features, labels

    print("Cache not found. Encoding dataset...")
    model, processor = get_CLIP_model()

    loader = torch.utils.data.DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=num_workers,
        pin_memory=True,
        shuffle=False,
    )

    feats_list: list[torch.Tensor] = []
    lbls_list:  list[torch.Tensor] = []
    n_encoded = 0
    n_total   = len(dataset)
    pad       = len(str(n_total))

    for imgs_batch, lbls_batch in loader:
        inputs = processor(images=list(imgs_batch), return_tensors="pt").to(device)
        e = _as_feature_tensor(model.get_image_features(**inputs))
        e = F.normalize(e, p=2, dim=-1)
        feats_list.append(e.cpu())
        lbls_list.append(lbls_batch)
        n_encoded += len(imgs_batch)
        print(f"Encoded {n_encoded:>{pad}}/{n_total} images ({100 * n_encoded / n_total:.1f}%)")

    features = torch.cat(feats_list, dim=0).to(device)
    labels   = torch.cat(lbls_list,  dim=0)

    torch.save({"features": features.cpu(), "labels": labels}, cache_path)
    print(f"Saved to {cache_path}. features: {tuple(features.shape)}, labels: {tuple(labels.shape)}")
    return features, labels

### Plotting utilities
Helper functions for visualizing retrieved images and their associated prompts, used across the notebook for consistent presentation of results.

In [ ]:
def plot_images(celeba_dataset: object, indices: list[int], n_cols: int, n_rows: int, figsize: tuple[int, int]=(20, 10)):
    """Utility function to plot a grid of images given their indices in the CelebA dataset.
    Args:
        celeba_dataset: The CelebA dataset object.
        indices: A list of indices corresponding to the images to be plotted.
        n_cols: Number of columns in the grid.
        n_rows: Number of rows in the grid.
        figsize: Size of the figure (width, height).
    """
    if len(indices) > n_cols * n_rows:
        raise ValueError("Number of indices exceeds the grid capacity")
    
    _, axes = plt.subplots(n_rows, n_cols, figsize=figsize)

    for counter, img_idx in enumerate(indices):
        img, _ = celeba_dataset[img_idx]
        if n_rows == 1:
            ax = axes[counter % n_cols]
        else:
            ax = axes[counter // n_cols, counter % n_cols]
        ax.imshow(img)
        ax.axis('off')

    plt.tight_layout()
    plt.show()

### Results visualization
Functions for visualizing the results of image retrieval, including displaying retrieved images alongside their prompts and similarity scores.

In [ ]:
def plot_metrics_across_k(average_results_per_query: list[dict], title: str = "Retrieval Metrics across K"):
    """
    Plot Recall@K and Precision@K as a grouped bar chart: one bar per query, grouped by K, with 95% confidence intervals.
    Args:
        average_results_per_query: Output from compute_query_average_results() for each query — list of per-query average dicts.
        title: Title for the overall figure.
    """
    k_values = [1, 5, 10]
    n_queries = len(average_results_per_query)
    x = np.arange(n_queries)
    width = 0.25
    offsets = [-width, 0.0, width]
    colors = [plt.cm.tab10(i) for i in range(len(k_values))]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(title)

    for k, offset, color in zip(k_values, offsets, colors):
        recall_means    = [q[f"Recall@{k}"]       for q in average_results_per_query]
        recall_cis      = [q[f"Recall@{k}_CI"]    for q in average_results_per_query]
        precision_means = [q[f"Precision@{k}"]    for q in average_results_per_query]
        precision_cis   = [q[f"Precision@{k}_CI"] for q in average_results_per_query]

        ax1.bar(x + offset, recall_means,    width, yerr=recall_cis,    capsize=4, ecolor="black", color=color, label=f"K={k}")
        ax2.bar(x + offset, precision_means, width, yerr=precision_cis, capsize=4, ecolor="black", color=color, label=f"K={k}")

    for ax, metric in [(ax1, "Recall"), (ax2, "Precision")]:
        ax.set_xlabel("Query")
        ax.set_ylabel(f"{metric}@K")
        ax.set_title(f"{metric}@K per query")
        ax.set_xticks(x)
        ax.set_xticklabels([f"Q{i+1}" for i in range(n_queries)])
        ax.set_ylim(0, 1)
        ax.grid(True, alpha=0.3, axis="y")
        ax.legend(title="K")

    plt.tight_layout()
    plt.show()


def plot_methods_comparison(method_results: dict[str, list[dict]], title: str = "Method Comparison across queries"):
    """
    Per-query line comparison of N retrieval methods. Layout is a 2 x 3 grid:
    rows are Recall (top) and Precision (bottom); columns are K in {1, 5, 10}.
    In each subplot, one segmented line per method connects the metric values
    over the query axis, so per-query differences between methods are visible.
    Args:
        method_results: Dict mapping method name -> average_results_per_query (same shape consumed by plot_metrics_across_k).
        title: Title for the overall figure.
    """
    k_values = [1, 5, 10]
    method_names = list(method_results.keys())
    n_methods = len(method_names)
    if n_methods == 0:
        raise ValueError("method_results must contain at least one method.")

    n_queries = len(next(iter(method_results.values())))
    x = np.arange(n_queries)
    colors = [plt.cm.tab10(i % 10) for i in range(n_methods)]

    fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True, sharey=True)
    fig.suptitle(title)

    for row_idx, metric in enumerate(["Recall", "Precision"]):
        for col_idx, k in enumerate(k_values):
            ax = axes[row_idx, col_idx]
            for method, color in zip(method_names, colors):
                ys = [q[f"{metric}@{k}"] for q in method_results[method]]
                ax.plot(x, ys, marker="o", color=color, label=method)
            ax.set_title(f"{metric}@{k}")
            ax.set_ylim(0, 1)
            ax.grid(True, alpha=0.3)
            ax.set_xticks(x)
            ax.set_xticklabels([f"Q{i+1}" for i in range(n_queries)], rotation=45, ha="right")

    for ax in axes[:, 0]:
        ax.set_ylabel("Score")
    for ax in axes[-1, :]:
        ax.set_xlabel("Query")

    axes[0, 0].legend(title="Method", loc="best")

    plt.tight_layout()
    plt.show()

## Data loading and Exploration
In this section, we load the dataset and perform an initial exploration to understand its structure and main characteristics.

The dataset used is `CelebA`, which contains 19,962 samples. Each sample consists of:
- an image of size `178 × 218`, representing a face of a celebrity;
- a set of 40 attributes that describe visual features of the person in the image.

We will briefly inspect the dataset by visualizing some samples and examining the associated attributes, in order to get a better understanding of the data before proceeding with further analysis.

In [ ]:
# Do *not* put `celeba` in the path.
# The dataset class will do that automatically!
data_root = Path("/content/datasets")
celeba = CelebA(root=data_root, split="test", download=False)

# This should be 19.962
print("Number of samples:", len(celeba))

# show element size
sample_img, sample_attrs = celeba[0]
print(f"Sample image size: {sample_img.size}")
print(f"Number of attributes: {len(sample_attrs)}")

### Sample visualization
First, we can visualize a random selection of images from the dataset to get a sense of the variety and quality of the images. We will display 50 random images in a grid format.

In [ ]:
# Get 50 random images and visualize them
indices = np.random.choice(len(celeba), size=50, replace=False)
plot_images(celeba, indices=indices, n_cols=10, n_rows=5)

### Attribute annotation
Now that we know how to load our dataset and we have visualized some samples, let's move to understanding how attributes are annotated in the dataset. Each image in the dataset is annotated with a set of 40 binary attributes, from the following list. 

Here, we also report how frequently each attribute appears in the dataset, which is important to understand the distribution of attributes and to design a retrieval system that can handle rare attributes effectively.


In [ ]:
all_labels = np.array([labels for _, labels in celeba])

attr_counts = all_labels.sum(axis=0)
attr_freq = all_labels.mean(axis=0)

print(f"{'Attribute':<20} {'Count':>10} {'Frequency':>10}")
print("-" * 45)

for attr, count, freq in zip(celeba.attr_names, attr_counts, attr_freq):
    print(f"{attr:<20} {count:>10} {freq:>10.3f}")

Let's define few other utilities functions that will facilitate the handling of attributes later on.
We will create a:
- `inx2attribute`: mapping from indices to attributes (e.g., index 0 corresponds to "5_o_Clock_Shadow", index 1 corresponds to "Arched_Eyebrows", etc.)

- `attribute2index`: mapping from attributes to indices (e.g., "5_o_Clock_Shadow" corresponds to index 0, "Arched_Eyebrows" corresponds to index 1, etc.)

- `retrieve_by_attributes(parameters)`: function that retrieves images based on specified attributes. This function will be crucial for our image retrieval system, allowing us to query the dataset for images that match certain attribute criteria.

- `plot_image_with_attributes`: function that displays an image along with its active attributes.

Query format for `retrieve_by_attributes`:
- Use `"+"` when the attribute must be present.
- Use `"-"` when the attribute must be absent.

Example: `{"Bald": "+", "Eyeglasses": "-"}`.

In [ ]:
# Assign a unique index to each attribute, and get the inverse mapping
idx2attribute = {idx: name for idx, name in enumerate(celeba.attr_names)}
attribute2idx = {name: idx for idx, name in enumerate(celeba.attr_names)}

def retrieve_by_attributes(parameters:dict):
    """
    Helper function that retrieve all the images that satisfy the conditions specified 
    in `parameters`.
    Params:
        - parameters: a dictionary where keys are attribute names and values are either "+" (must have the attribute) or "-" (must not have the attribute).  
    Returns:
        - A list of indices of images that satisfy the specified conditions.
    """
    # Start with all indices
    valid_indices = set(range(len(celeba)))
    
    # For each attribute condition, filter the indices
    for attr_name, value in parameters.items():
        attr_idx = attribute2idx[attr_name]
        if value == "+":
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 0:
                    valid_indices.remove(idx)
        elif value == "-":
            # Must not have the attribute
            for idx in valid_indices.copy():
                if celeba[idx][1][attr_idx] == 1:
                    valid_indices.remove(idx)
        else:
            raise ValueError(f"Invalid value for attribute condition: {value}. Use '+' or '-'.")
    
    return list(valid_indices)

def plot_image_with_attributes(idx: int, figsize: tuple[int, int]=(10, 5)):
    """Helper function to plot a single image along with its active attributes as text on the right side of the image."""
    img, labels = celeba[idx]
    active_attrs = [idx2attribute[idx] for idx, value in enumerate(labels) if value == 1]

    fig, (ax_img, ax_text) = plt.subplots(1, 2, figsize=figsize)
    # Left: image
    ax_img.imshow(img)
    ax_img.axis('off')

    # Right: centered text
    ax_text.axis('off')
    text = "\n".join(active_attrs)

    ax_text.text(
        0.5, 0.5, text,
        fontsize=10,
        ha='center',   # horizontal alignment
        va='center'    # vertical alignment
    )

    plt.tight_layout()
    plt.show()

Now that we have the mapping, we can easily get the attributes of any image in the dataset. For example, let's get the attributes of a given image index.

In [ ]:
IMAGE_INDEX = 99
plot_image_with_attributes(99)

Now that we have everything in place, let's try to analyze some possible queries.


In [ ]:
query_1 = {"Bald": "+",
           "Smiling": "+",
           "Eyeglasses": "-",
           }
retrieved_images = retrieve_by_attributes(query_1)
print(f"Number of retrieved images: {len(retrieved_images)}")

# Plot up to 10 random retrieved images (without replacement).
n_samples = min(10, len(retrieved_images))
if n_samples == 0:
    print("No images match this query.")
else:
    sampled_indices = np.random.choice(retrieved_images, size=n_samples, replace=False)
    n_cols = 5
    n_rows = int(np.ceil(n_samples / n_cols))
    plot_images(celeba, indices=sampled_indices, n_cols=n_cols, n_rows=n_rows)

# Offline Feature Extraction
In this section we extract features from our dataset using a Vision Language Model (VLM).

This representation is computed once and then kept frozen offline. The goal is not to improve the VLM encoder itself, but to study and improve retrieval behavior on top of fixed CLIP embeddings.

In [ ]:
EVALUATION_CACHE_DIR = "/content/drive/MyDrive/datasets/clip_cache"
EVALUATION_CACHE_PATH = os.path.join(EVALUATION_CACHE_DIR, "embeddings.pt")

### Extracting features using a VLM
In this section we will be using the CLIP model to extract features from our dataset. We will be using the ViT-B/32 model, which is a smaller version of the original CLIP model. 

Since these embeddings are static, we can compute them offline and keep them frozen. This means that we don't have to worry about the computational cost of computing the embeddings during training, and we can also use a larger batch size for training our retriever model.

The result of this process will be a list of embeddings, where each embedding is of size 512 and corresponds to an image in our dataset. 

In [ ]:
# Get the encoded dataset, using cached features if available
embeddings, embedding_labels = get_encoded_dataset(celeba, device, EVALUATION_CACHE_PATH, batch_size=128)


## Sanity check
Now that embeddings for the image dataset are available, let's run a quick sanity check to verify retrieval quality.

We will pick a source image, compare its CLIP embedding against all dataset embeddings, and inspect the nearest matches.

In [ ]:
source_idx = 10006
img, _ = celeba[source_idx]

plt.figure(figsize=(4, 4))
plt.axis('off')
plt.imshow(img)

Now that we have our source image and its CLIP encoding, let's find the nearest embeddings in the dataset.

We normalize embeddings to unit vectors at extraction time, so similarity between any two embeddings reduces to a plain dot product.
For unit vectors, `dot(a, b) == cosine_similarity(a, b)`, so the metric is unchanged — but retrieval becomes a single matrix multiply instead of a per-pair loop.
We exclude the source image itself from the top results.

In [ ]:
if "embeddings" not in globals():
    raise RuntimeError(
        "Embeddings not found. Run the offline feature extraction cell above first."
    )

source_embedding = embeddings[source_idx]

# Dot product == cosine similarity for unit-norm embeddings (single matrix-vector multiply)
# 19962 x 512 @ 512 -> 19962, all on GPU.
similarities = embeddings @ source_embedding

# Get the 6 highest-similarity matches and drop the source itself.
top_vals, top_idx = torch.topk(similarities, k=6)
nearest_indices = top_idx[1:].tolist()         # Python ints needed to index `celeba[...]`
nearest_similarities = top_vals[1:].tolist()   # Python floats needed for plot titles

print("Nearest indices:", nearest_indices)
print("Nearest cosine similarities:", nearest_similarities)


We have found the nearest embeddings! Let's visualize the nearest images to our source image and see if they are indeed similar.

In [ ]:
fig, axes = plt.subplots(ncols=5, figsize=(25, 5))

for i, image_idx in enumerate(nearest_indices):
    img, labels = celeba[image_idx]
    ax = axes[i]
    
    ax.set_title(f"Cosine sim: {nearest_similarities[i]:.4f}")
    
    ax.imshow(img)
    ax.axis('off')

plt.tight_layout()
plt.show()

## Embedding Analysis: Class × Image Cosine Heatmap

Before introducing any retrieval method, it is useful to inspect how well CLIP separates the 40 CelebA attributes on real images.

For every attribute we:
1. Encode the naturalized prompt through the CLIP text encoder.
2. Pick **one "pure" image** from the dataset: an image where this attribute is active and the count of other co-active attributes is minimal. This makes the diagonal of the resulting heatmap interpretable as a true match while reducing visual confusion from CelebA's many overlapping labels.
3. Build the 40×40 matrix of cosine similarities between the 40 text embeddings and the 40 selected image embeddings.

If CLIP is well-aligned with these attribute concepts, the heatmap diagonal should dominate each row. Strong off-diagonal cells highlight attribute pairs CLIP confuses on this dataset (e.g. `Wavy_Hair` vs `Straight_Hair`, `Heavy_Makeup` vs `Wearing_Lipstick`).

In [ ]:
def _select_pure_image_idxs(all_labels: np.ndarray, rng: np.random.Generator) -> list[int]:
    """For each attribute, pick one image where it is positive and the count of other positive attributes is minimal."""
    selected = []
    for attr_idx in range(all_labels.shape[1]):
        candidates = np.where(all_labels[:, attr_idx] == 1)[0]
        if len(candidates) == 0:
            selected.append(int(rng.integers(0, all_labels.shape[0])))
            continue
        other_counts = all_labels[candidates].sum(axis=1) - 1
        purest = candidates[other_counts == other_counts.min()]
        selected.append(int(rng.choice(purest)))
    return selected


def _plot_cosine_heatmap(cos_mat: np.ndarray, attr_names: list[str]) -> None:
    """Plot a cosine similarity heatmap of text prompts vs sampled images."""
    n = len(attr_names)
    fig, ax = plt.subplots(figsize=(14, 14))
    im = ax.imshow(cos_mat, cmap="viridis", aspect="equal")
    ticks = np.arange(n)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.set_xticklabels(attr_names, rotation=90, fontsize=8)
    ax.set_yticklabels(attr_names, fontsize=8)
    ax.set_xlabel("Sampled image (chosen as 'pure' positive for this attribute)")
    ax.set_ylabel("Text prompt: 'A picture of a person with {attr}'")
    ax.set_title(f"CLIP cosine similarity: {n} attribute prompts × {n} sampled images")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="cosine similarity")
    plt.tight_layout()
    plt.show()


def _print_cosine_diagnostics(cos_mat: np.ndarray) -> None:
    """Print diagonal vs off-diagonal cosine similarity statistics."""
    n = cos_mat.shape[0]
    diag = np.diag(cos_mat)
    off_diag_mean = (cos_mat.sum() - diag.sum()) / (cos_mat.size - n)
    diag_argmax_rate = float((cos_mat.argmax(axis=1) == np.arange(n)).mean())
    print(f"Mean diagonal cosine:      {diag.mean():.4f}")
    print(f"Mean off-diagonal cosine:  {off_diag_mean:.4f}")
    print(f"Diagonal-argmax rate:      {diag_argmax_rate:.2%} (attributes where the matching image is the row's argmax)")


attr_names = list(celeba.attr_names)
prompts = [f"{name.replace('_', ' ').lower()}" for name in attr_names]
text_embs = encode_texts(prompts, model, processor, device)  # (n_attrs, 512)


rng = np.random.default_rng(seed=0)
selected_idxs = _select_pure_image_idxs(all_labels, rng)
selected_img_embs = embeddings[selected_idxs].to(text_embs.device)  # (n_attrs, 512)


cos_mat = (text_embs @ selected_img_embs.T).detach().cpu().numpy()  # (n_attrs, n_attrs)
_print_cosine_diagnostics(cos_mat)

In [ ]:
_plot_cosine_heatmap(cos_mat, attr_names)

## Metrics
In this section we will define the metrics that we will use to evaluate our retrieval system.

In particular, we will be using the following metrics:
- **Recall@K**: This metric measures if **at least one** valid truth image is present in the top K retrieved images. It is defined as:
$$Recall@K = \begin{cases} 1 & \text{if } |\mathcal{R}_K \cap \mathcal{G}| > 0 \\ 0 & \text{otherwise} \end{cases}$$
- **Precision@K**: This metric measures the proportion of relevant images in the top K retrieved images. It is defined as:
$$Precision@K = \frac{| \mathcal{R}_K \cap \mathcal{G} |}{K}$$

Where:
- $\mathcal{R}_K$ is the set of top K retrieved images for the given query.
- $\mathcal{G}$ is the set of ground truth relevant images for the given query.



In [ ]:
def evaluate_retrieval(
    retrieved_indices: list[int],
    ground_truth_indices: list[int],
    k: int
) -> dict:
    """
    Evaluate the retrieval performance for a single source image.
    Args:
        - retrieved_indices: list of image IDs predicted by the model, ordered by similarity (descending).
        - ground_truth_indices: list of valid target IDs from the benchmark JSON.
        - k: the cutoff for top-K evaluation (e.g., 1, 5, 10).
    Return:
        - A dictionary containing Recall@K and Precision@K metrics.

    """
    # Get the top K retrieved indices
    #NOTE: the retrieved_indices must be ordered by similarity in descending order
    top_k_retrieved = retrieved_indices[:k]

    # Calculate the intersection between predictions and ground truth
    hits = set(top_k_retrieved).intersection(set(ground_truth_indices))
    num_hits = len(hits)

    # Metrics calculations
    # Recall@K (Hit Rate): 1 if at least one match is found, 0 otherwise
    recall_at_k = 1 if num_hits > 0 else 0

    # Precision@K: Fraction of top K predictions that are correct
    precision_at_k = num_hits / k

    return {
        f"Recall@{k}": recall_at_k,
        f"Precision@{k}": precision_at_k
    }

In [ ]:
# --- Example Usage ---
# Suppose the model returns these indices from most to least similar:
predictions = [1, 2, 3, 4, 5]
# And we load this from our JSON for this specific source:
ground_truth = [3, 2, 1]

# Evaluate at K=1 and K=5
print("Results @ 1:", evaluate_retrieval(predictions, ground_truth, k=1))
print("Results @ 5:", evaluate_retrieval(predictions, ground_truth, k=5))

## Evaluation Protocol
To assess the performance of our retrieval system, we utilize a standardized benchmark of queries stored in a JSON file. Each entry in the dataset follows this structure:

* **`query`**: A string representing the textual modification (e.g., `"+glasses, -smiling"`).
* **`ground_truth`**: A dictionary where:
    * **Keys** are the indices of the **source images** used as the starting point.
    * **Values** are lists of indices for images considered valid retrievals for that specific source.

### Example Structure
```json
{
    "query": "+glasses, -smiling",
    "ground_truth": {
        "0": [1, 2, 3],
        "4": [5, 6, 7]
    }
}
```
In this example, image 0 serves as a source image (e.g., a smiling person without glasses). The system is expected to retrieve images 1, 2, or 3, which represent the "target" state (a non-smiling person with glasses), which should be visually similar to the source image but with the specified modifications applied.

In [ ]:
# Open the JSON file containing the benchmark annotations
annotations_path = Path("/content/drive/MyDrive/datasets/celeba_evaluation.json")
with open(annotations_path, "r") as f:
    annotations = json.load(f)

# Print the number of annotations loaded
print(f"Loaded {len(annotations)} queries!")

We can define some utility functions to facilitate the evaluation process

In [ ]:
# Display a sample annotation to understand the structure of the data
print("Sample annotation shape", annotations[0].keys())

# Extract and print first text query
print("Text-Query example:", annotations[0].get("query", ""))

# Extract and print the source image ID for the first annotation
print("Source-Image example:", list(annotations[0].get("ground_truth", {}).keys())[:5],"...")

# Extract and print the list of ground truth indices for the first annotation
print("List of ground truth indices for the first annotation:", annotations[0].get("ground_truth", {}).get("13", [])[:5], "...")

def get_text_query(annotation: dict) -> str:
    """
    Helper function to extract the text query from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
    Returns:
        - A string representing the text query (e.g., "+glasses, -smile").
    """
    return annotation.get("query", "")

def get_source_image_idxs(annotation: dict) -> list[int]:
    """
    Helper function to extract the source image IDs from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
    Returns:
        - A list of integers representing the source image IDs.
    """    
    # The key of the "ground_truth" must be converted to int since JSON keys are always strings
    return [int(key) for key in annotation.get("ground_truth", {}).keys()]

def get_ground_truth_indices(annotation: dict, source_image_idx: int) -> list[int]:
    """
    Helper function to extract the list of valid target IDs from a benchmark annotation.
    Args:
        - annotation: A dictionary containing the benchmark annotation for a single query.
        - source_image_idx: The index of the source image for which to retrieve ground truth indices.
    Returns:
        - A list of valid target IDs (integers) that are considered correct matches for the given query.
    """
    return annotation.get("ground_truth", {}).get(str(source_image_idx), [])


In [ ]:
# Let's test these utility functions on the first annotation in the dataset
annotation = annotations[1]

text_query = get_text_query(annotation)
print("Text query:", text_query )

source_image_idx = get_source_image_idxs(annotation)[0]
print("Source image index:", source_image_idx)
plot_image_with_attributes(source_image_idx, figsize=(4, 4))

# Get the first 5 ground truth indices for this annotation and source image
ground_truth_indices = get_ground_truth_indices(annotation, source_image_idx)[:5]
print("Ground truth indices for this query:", ground_truth_indices)

plot_images(celeba, indices=ground_truth_indices, n_cols=5, n_rows=1, figsize=(10, 2))

### Evaluation Function

We evaluate the retrieval performance of each fusion mechanism on the benchmark dataset, comparing it against the baseline method.

We compute the recall and precision metrics for each source image in the query for `"K = {1, 5, 10}"`.
Then we average the result across all source images and keep track on each query separately.

In [ ]:
def evaluate( 
        fusion_mechanism: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
        query_embedding_function: Callable[[str, torch.nn.Module, CLIPProcessor, torch.device], list[torch.Tensor]],
        annotations: list[dict],
        model: torch.nn.Module,
        processor: CLIPProcessor,
        device: torch.device,
        embeddings: torch.Tensor
    ) -> list[dict]:
    '''
    Evaluates the retrieval performance of a given fusion mechanism on the benchmark dataset.
    Args:
    - fusion_mechanism: A function that takes in a text embedding and an image embedding and returns a unified query embedding (e.g., by summing them together).

    - query_embedding_function: A function that takes in a text query and returns its embedding using the CLIP model.
    - annotations: A list of benchmark annotations, where each annotation contains a text query, source image index, and a list of valid target indices.
    - model: The CLIP model to be used for computing text embeddings.
    - processor: The CLIP processor to be used for preprocessing text queries.
    - device: The device (CPU or GPU) on which the model is located.
    - embeddings: A tensor containing the pre-computed CLIP image embeddings for the entire dataset
    Returns:
    - A list of evaluation results for each query in the benchmark dataset, where each result contains the all list of evaluation metrics for each source image of the query.
    '''
    evaluation_results = []

    # For each annotation in the benchmark dataset
    for i, annotation in enumerate(annotations):
        print(f"Evaluating query Q{i+1}: {annotation.get('query', '')}")

        # Extract the text query and compute its embedding using the provided query_embedding_function
        text_query = get_text_query(annotation)

        embedded_text_query = query_embedding_function(text_query, model, processor, device)

        # Get the source image indices for this annotation
        query_image_idxs = get_source_image_idxs(annotation)

        query_evaluation_results = {}
        # Evaluate the retrieval performance for each source image
        for query_image_idx in query_image_idxs:
            
            # Compute the fused embedding
            embedded_query_image = embeddings[query_image_idx]
            fused_embedding = fusion_mechanism(embedded_text_query, embedded_query_image)
            
            # Normalize the fused embedding 
            fused_embedding = torch.nn.functional.normalize(fused_embedding, dim=0)

            # Compute the cosine similarity between the fused embedding and all image embeddings in the dataset
            # Equal to dot product since all embeddings are normalized to unit sphere
            similarities = torch.matmul(embeddings, fused_embedding)

            # Get the top 10 most similar images, excluding the source image itself
            _, topk_indices = torch.topk(similarities, k=11)
            topk_indices = topk_indices[topk_indices != query_image_idx][:10]
            retrieved_indices = topk_indices.tolist()

            # Evaluate the retrieval performance at K=1, 5, and 10 using the provided evaluation function
            image_query_eval_metrics = {}
            for k in [1, 5, 10]:
                eval_metrics = evaluate_retrieval(
                    retrieved_indices=retrieved_indices,
                    ground_truth_indices=get_ground_truth_indices(annotation, query_image_idx),
                    k=k
                )
                image_query_eval_metrics[k] = eval_metrics
            # Store the evaluation results for this source image
            query_evaluation_results[query_image_idx] = image_query_eval_metrics

        # Compute the average evaluation results across all source images for this query and store it in the final evaluation results list        
        evaluation_results.append(query_evaluation_results)
    return evaluation_results
        

def compute_query_average_results(query_evaluation_results: dict) -> dict:
    '''
    Computes the average Recall@K and Precision@K across all source images of a query for K=1, 5, and 10.
    Args:
    - query_evaluation_results: A dict mapping source image index to its evaluation metrics for K=1, 5, and 10.
    Returns:
    - A dictionary containing the average results for the query
    '''
    average_results = {}
    
    for k in [1, 5, 10]:
        # Compute the sum of Recall@K and Precision@K across all source images for this query
        recall_sum = 0
        precision_sum = 0
        # Get the number of test that we run for this query
        num_images = len(query_evaluation_results)

        # Iterate over the evaluation results for each source image and accumulate the Recall@K and Precision@K values
        for _, eval_metrics_per_k in query_evaluation_results.items():
            recall_sum += eval_metrics_per_k[k][f"Recall@{k}"]
            precision_sum += eval_metrics_per_k[k][f"Precision@{k}"]
        # Compute the average Recall@K and Precision@K for this query and store it in the average_results dictionary
        average_results[f"Recall@{k}"] = recall_sum / num_images
        average_results[f"Precision@{k}"] = precision_sum / num_images
        
        # Compute the 95% confidence intervals for Recall@K and Precision@K
        recall_std_error = np.sqrt((average_results[f"Recall@{k}"] * (1 - average_results[f"Recall@{k}"])) / num_images)
        precision_std_error = np.sqrt((average_results[f"Precision@{k}"] * (1 - average_results[f"Precision@{k}"])) / num_images)
        average_results[f"Recall@{k}_CI"] = 1.96 * recall_std_error
        average_results[f"Precision@{k}_CI"] = 1.96 * precision_std_error
    
    return average_results

## Baseline Method
To establish a baseline for our retrieval system, we evaluate a **zero-shot, training-free approach** that relies exclusively on CLIP embeddings and cosine similarity. 

The process is as follows:
* **Encode the source image** using the CLIP Vision Encoder to obtain its visual embedding.
* **Encode the query attributes** using the CLIP Text Encoder to obtain a text-based embedding.
* **Sum the two embeddings** to create a unified query representation:
* **Rank results** by computing the cosine similarity between this combined embedding and all candidate embeddings in the dataset.

In [ ]:
def baseline_fusion(normalized_text_embedding: list[torch.Tensor], normalized_image_embedding: torch.Tensor, ALPHA: float = 1.0, BETA: float = 1.0) -> torch.Tensor:
    """
    Combine the query text and source image embeddings into a single query embedding.
    Parameters:
        - normalized_text_embedding: A list of normalized text embeddings for each component of the text query
        - normalized_image_embedding: A normalized image embedding for the source image
        - ALPHA: The weight for the text embedding in the fusion mechanism
        - BETA: The weight for the image embedding in the fusion mechanism
    Return:
        - A single query embedding that combines the text and image embeddings.
    """
    
    # If the text query has multiple components, we sum their embeddings together to get a single text embedding.
    if isinstance(normalized_text_embedding, list):
        normalized_text_embedding = torch.stack([embedding.view(-1) for embedding in normalized_text_embedding]).sum(dim=0)
    else:
        # If it's already a single embedding, we just need to flatten it to 1D
        normalized_text_embedding = normalized_text_embedding.view(-1)

    # Return the weighted sum of the normalized text embedding and the normalized image embedding as the final query embedding
    return ALPHA * normalized_text_embedding + BETA * normalized_image_embedding.view(-1)

def baseline_embed_query(text_query: str, model: torch.nn.Module, processor, device) -> list[torch.Tensor]:
    """Encode the full text query into a single normalized 1D tensor."""
    return [encode_text(text_query, model, processor, device)]

### Baseline Evaluation

In [ ]:
evaluation_results_baseline = evaluate(
    fusion_mechanism=baseline_fusion,
    query_embedding_function=baseline_embed_query,
    annotations=annotations,
    model=model,
    processor=processor,
    device=device,
    embeddings=embeddings
)
# Compute the average evaluation for each query
average_results_per_query_baseline = [compute_query_average_results(query_result) for query_result in evaluation_results_baseline]

In [ ]:
plot_metrics_across_k(average_results_per_query_baseline, title="Baseline Fusion Performance across K")

## Method 2 — Per-Attribute Logit Composition (training-free)

The previous "weighted fusion" approach was just a hyperparameter sweep on top of the baseline (`α·text + β·image`). It carries two known weaknesses:

1. **Source-image leakage.** Adding the source CLIP embedding to the query injects every attribute already present in the source — including the ones we want to *change* — and pulls the result toward look-alikes of the source.
2. **Embedding-space negation.** Multiplying an attribute embedding by `-1` is geometrically odd in CLIP's space: a unit vector negated points to a region that is *not* the linguistic complement of the concept, and summing several negated unit vectors compounds the issue.

We replace the fusion with a **per-attribute logit composition** that is the principled framing for multi-label compositional retrieval:

$$
\text{score}(\mathbf{x}, q) = \sum_{i \in q} s_i \cdot \cos\big(\mathbf{e}_{\text{img}}(\mathbf{x}), \mathbf{e}_{\text{txt}}(a_i)\big)
$$

where `s_i ∈ {+1, -1}` is the sign of attribute `a_i` in the query, and `e_txt(a_i)` is the CLIP text embedding of a single, fixed prompt per attribute (precomputed once for all 40 attributes).

Sign-flipping now lives in **similarity space**, not embedding space — this matches the standard zero-shot multi-label CLIP setup and avoids both pitfalls above. The source image embedding is no longer used at query time.

In [ ]:
# Precompute one text embedding per attribute, once, for the whole notebook.
# Uses `encode_texts` with the same bare-name prompts as the heatmap cell, so
# the two views of attribute-text geometry stay aligned.

# Source `n_attrs` from the (N, n_attrs) numpy matrix built directly from celeba,
# not from the cached `labels` tensor (whose shape can vary across older caches).
_all_labels_np = np.asarray(all_labels)
if _all_labels_np.ndim != 2:
    raise RuntimeError(
        f"`all_labels` is not 2-D (got shape {_all_labels_np.shape}). "
        "Re-run the Attribute Annotation cell."
    )
n_attrs = _all_labels_np.shape[1]
attr_names = list(celeba.attr_names)[:n_attrs]


@torch.no_grad()
def precompute_attribute_text_embeddings() -> torch.Tensor:
    """Return a (n_attrs, 512) L2-normalized matrix of CLIP text embeddings,
    one row per attribute. Uses the same bare-name prompts as the cosine-heatmap
    diagnostics cell, so the two views of attribute-text geometry stay aligned."""
    prompts = [name.replace("_", " ").lower() for name in attr_names]
    return encode_texts(prompts, model, processor, device)   # (n_attrs, D), L2-normalized per row

ATTR_TEXT_EMBS = precompute_attribute_text_embeddings().to(embeddings.device)  # (n_attrs, 512)


def parse_signed_query(text_query: str) -> torch.Tensor:
    """Parse a query like '+Bald, +Smiling, -Eyeglasses' into a signed weight vector of length n_attrs.

    Returns a torch.Tensor of shape (n_attrs,), with +1 / -1 / 0 entries.
    """
    w = torch.zeros(n_attrs, device=embeddings.device)
    for component in text_query.split(","):
        component = component.strip()
        if not component:
            continue
        sign_char, attr_name = component[0], component[1:].strip()
        if attr_name not in attribute2idx:
            raise KeyError(f"Unknown attribute '{attr_name}' in query '{text_query}'")
        idx = attribute2idx[attr_name]
        if idx >= n_attrs:
            continue   # attribute exists in the name list but not in the labels matrix
        sign = 1.0 if sign_char == "+" else -1.0 if sign_char == "-" else 0.0
        w[idx] = sign
    return w


def evaluate_logit_composition(
    annotations: list[dict],
    embeddings: torch.Tensor,
    attr_text_embs: torch.Tensor,
) -> list[dict]:
    """Score every candidate image by sum_i sign_i * cos(img, attr_text_embs[i]).

    The source image embedding is intentionally not used: this is a pure text-driven retrieval.
    Returns the same nested structure produced by `evaluate(...)`.
    """
    # Precompute the (N_imgs, n_attrs) attribute logit matrix once. Each entry is
    # cos(img_n, attr_i), since both sides are L2-normalized.
    img_attr_logits = embeddings @ attr_text_embs.T  # (N, n_attrs)

    evaluation_results = []
    for i, annotation in enumerate(annotations):
        print(f"Evaluating query Q{i+1}: {annotation.get('query', '')}")
        w = parse_signed_query(get_text_query(annotation))     # (n_attrs,)
        scores = img_attr_logits @ w                            # (N,)

        query_image_idxs = get_source_image_idxs(annotation)
        query_evaluation_results = {}
        for query_image_idx in query_image_idxs:
            _, topk_indices = torch.topk(scores, k=11)
            topk_indices = topk_indices[topk_indices != query_image_idx][:10]
            retrieved_indices = topk_indices.tolist()

            image_query_eval_metrics = {}
            for k in [1, 5, 10]:
                image_query_eval_metrics[k] = evaluate_retrieval(
                    retrieved_indices=retrieved_indices,
                    ground_truth_indices=get_ground_truth_indices(annotation, query_image_idx),
                    k=k,
                )
            query_evaluation_results[query_image_idx] = image_query_eval_metrics

        evaluation_results.append(query_evaluation_results)
    return evaluation_results


In [ ]:
evaluation_results_logit = evaluate_logit_composition(
    annotations=annotations,
    embeddings=embeddings,
    attr_text_embs=ATTR_TEXT_EMBS,
)
# Per-query average results, same shape as the other methods
average_results_per_query_logit = [
    compute_query_average_results(query_result) for query_result in evaluation_results_logit
]

plot_metrics_across_k(
    average_results_per_query_logit,
    title="Per-Attribute Logit Composition (training-free) — Performance across K",
)

## Method 3 — Prompt Ensembling v2 (training-free)

The first prompt-ensembling attempt underperformed for two reasons that are now visible thanks to Method 2:

1. **Sign-flip negation in embedding space.** Same failure mode as the weighted fusion: `-e+` does not point at "person without smile" in CLIP space. Method 2 already drops this in favor of similarity-space sign-flip.
2. **Sum-of-embeddings aggregation.** Adding several L2-normalized phrase embeddings together is a noisy proxy for "this image matches all of these attributes". Per-attribute logit composition (Method 2) is the clean version.

Method 3 keeps the logit-composition framing of Method 2 and improves the **per-attribute text embeddings** themselves:

- **3a. Expanded template bank.** CLIP's full 80-template prompt ensemble (the canonical ImageNet recipe) plus a handful of portrait-specific templates.
- **3b. Linguistic negation, not sign-flip.** For each attribute we build *two* embeddings via the ensemble: `e⁺` = "a person with {attr}" and `e⁻` = "a person without {attr}". At retrieval, a `+attr` term contributes `cos(img, e⁺)` and a `-attr` term contributes `cos(img, e⁻)` — the negative is a real negative description, not the negation of the positive vector.
- **3c. Same logit-composition aggregation as Method 2.** This is the strict generalization of Method 2: Method 2 sets `e⁻_i = -e⁺_i` implicitly via the sign; Method 3 learns a separate linguistic `e⁻_i` and scores it on its own.

We expect a clear gain over both old Method 3 (which mixed sign-flip with a heavy ensemble) and Method 2 (which uses a single short prompt per attribute).

In [ ]:
# Person-referring positive AND negative phrases for each CelebA attribute.
# The previous version only stored positives and negated their embedding by -1;
# we now also store linguistic negatives so the negative side of the score is
# computed against an actual "without ..." description.
humanized_mappings_pos = {
    "5_o_Clock_Shadow":     ["a person with a 5 o'clock shadow", "a person with light facial stubble", "a person with short beard stubble", "a face with a 5 o'clock shadow", "a man with a 5 o'clock shadow", "a person with visible beard stubble"],
    "Arched_Eyebrows":      ["a person with arched eyebrows", "a person with curved eyebrows", "a face with high arched eyebrows", "a portrait with strongly arched eyebrows", "a person whose eyebrows are clearly arched"],
    "Attractive":           ["an attractive person", "a good-looking person", "a visually appealing person", "a beautiful person", "an attractive face", "a strikingly attractive person"],
    "Bags_Under_Eyes":      ["a person with bags under the eyes", "a person with eye bags", "a tired-looking person with under-eye bags", "a face with visible under-eye bags", "a portrait of a person with bags under the eyes"],
    "Bald":                 ["a bald person", "a person with no hair", "a person with a shaved head", "a person with a completely bald head", "a man who is bald", "a portrait of a bald person"],
    "Bangs":                ["a person with bangs", "a person with fringe hair", "a face with bangs across the forehead", "a portrait of someone with bangs", "a person whose hair has bangs"],
    "Big_Lips":             ["a person with full lips", "a person with big lips", "a face with prominent lips", "a portrait of a person with very full lips"],
    "Big_Nose":             ["a person with a big nose", "a person with a large nose", "a face with a prominent nose", "a portrait of a person with a noticeably big nose"],
    "Black_Hair":           ["a person with black hair", "a person with dark black hair", "a portrait of a black-haired person", "a person whose hair is black"],
    "Blond_Hair":           ["a person with blond hair", "a person with blonde hair", "a person with light blonde hair", "a portrait of a blond person", "a person whose hair is blonde"],
    "Blurry":               ["a blurry photo of a person", "an out-of-focus image of a person", "a blurred image of a person", "a low-quality blurry portrait", "a defocused photograph of a face"],
    "Brown_Hair":           ["a person with brown hair", "a person with dark brown hair", "a portrait of a brown-haired person", "a person whose hair is brown"],
    "Bushy_Eyebrows":       ["a person with bushy eyebrows", "a person with thick eyebrows", "a face with very thick eyebrows", "a portrait of a person with bushy eyebrows"],
    "Chubby":               ["a chubby person", "a person with a round face", "a person with a chubby face", "a portrait of a chubby person"],
    "Double_Chin":          ["a person with a double chin", "a person with a noticeable double chin", "a face with a clear double chin", "a portrait of a person with a double chin"],
    "Eyeglasses":           ["a person wearing eyeglasses", "a person wearing glasses", "a person with glasses", "a face with eyeglasses", "a portrait of a person wearing glasses", "a person who wears glasses"],
    "Goatee":               ["a person with a goatee", "a person with a goatee beard", "a man with a goatee", "a portrait of a person with a goatee"],
    "Gray_Hair":            ["a person with gray hair", "a person with grey hair", "a person with silver hair", "a portrait of a gray-haired person", "an older person with gray hair"],
    "Heavy_Makeup":         ["a person wearing heavy makeup", "a person with noticeable makeup", "a person with strong makeup", "a face with heavy makeup", "a portrait of a person wearing heavy makeup"],
    "High_Cheekbones":      ["a person with high cheekbones", "a person with prominent cheekbones", "a face with sharply defined high cheekbones"],
    "Male":                 ["a man", "a male person", "a portrait of a man", "a photograph of a man", "a male face"],
    "Mouth_Slightly_Open":  ["a person with their mouth slightly open", "a person with slightly open lips", "a face with parted lips", "a portrait of a person whose mouth is slightly open"],
    "Mustache":             ["a person with a mustache", "a person with facial hair and a mustache", "a man with a mustache", "a portrait of a person with a mustache"],
    "Narrow_Eyes":          ["a person with narrow eyes", "a person with small eyes", "a face with narrow eyes", "a portrait of a person with narrow eyes"],
    "No_Beard":             ["a clean-shaven person", "a person without a beard", "a person with no facial hair", "a portrait of a clean-shaven person", "a face without any beard"],
    "Oval_Face":            ["a person with an oval face", "a person with an oval-shaped face", "a portrait of a person with an oval face"],
    "Pale_Skin":            ["a person with pale skin", "a person with light skin tone", "a portrait of a person with pale skin", "a face with very pale skin"],
    "Pointy_Nose":          ["a person with a pointy nose", "a person with a sharp nose", "a face with a pointy nose"],
    "Receding_Hairline":    ["a person with a receding hairline", "a person with thinning hairline", "a portrait of a person whose hairline is receding"],
    "Rosy_Cheeks":          ["a person with rosy cheeks", "a person with flushed cheeks", "a face with rosy cheeks"],
    "Sideburns":            ["a person with sideburns", "a person with long sideburns", "a face with sideburns"],
    "Smiling":              ["a smiling person", "a person who is smiling", "a person with a happy expression", "a person with a big smile", "a portrait of a smiling person", "a face with a smile"],
    "Straight_Hair":        ["a person with straight hair", "a person with smooth straight hair", "a portrait of a person with straight hair"],
    "Wavy_Hair":            ["a person with wavy hair", "a person with curly wavy hair", "a portrait of a person with wavy hair"],
    "Wearing_Earrings":     ["a person wearing earrings", "a person with earrings", "a portrait of a person wearing earrings"],
    "Wearing_Hat":          ["a person wearing a hat", "a person with a hat", "a portrait of a person wearing a hat"],
    "Wearing_Lipstick":     ["a person wearing lipstick", "a person with lipstick", "a portrait of a person wearing lipstick"],
    "Wearing_Necklace":     ["a person wearing a necklace", "a person with a necklace", "a portrait of a person wearing a necklace"],
    "Wearing_Necktie":      ["a person wearing a necktie", "a person with a tie", "a portrait of a person wearing a necktie"],
    "Young":                ["a young person", "a youthful person", "a person who looks young", "a portrait of a young person", "a young-looking face"],
}

# Linguistic negatives. We avoid the "not X" construction wherever possible because
# CLIP's text encoder attends to the object token regardless of the "not" — phrasing
# matters. Where a clean linguistic opposite exists (e.g. clean-shaven vs bearded)
# we use it; otherwise we lean on "without {attr}" / "no {attr}" framings.
humanized_mappings_neg = {
    "5_o_Clock_Shadow":     ["a clean-shaven person", "a person with no facial stubble", "a person without a 5 o'clock shadow", "a smoothly shaven face"],
    "Arched_Eyebrows":      ["a person with flat eyebrows", "a person whose eyebrows are not arched", "a face with straight eyebrows"],
    "Attractive":           ["an unattractive person", "a plain-looking person", "an ordinary-looking person"],
    "Bags_Under_Eyes":      ["a person without bags under the eyes", "a person with no eye bags", "a fresh-looking face without under-eye bags"],
    "Bald":                 ["a person with hair", "a person with a full head of hair", "a person who is not bald"],
    "Bangs":                ["a person without bangs", "a person with no fringe", "a face without bangs"],
    "Big_Lips":             ["a person with thin lips", "a person with small lips", "a person without big lips"],
    "Big_Nose":             ["a person with a small nose", "a person without a big nose", "a face with a small nose"],
    "Black_Hair":           ["a person without black hair", "a person whose hair is not black"],
    "Blond_Hair":           ["a person without blond hair", "a person whose hair is not blonde"],
    "Blurry":               ["a sharp clear photo of a person", "a high quality in-focus portrait", "a crisp clear image of a face"],
    "Brown_Hair":           ["a person without brown hair", "a person whose hair is not brown"],
    "Bushy_Eyebrows":       ["a person with thin eyebrows", "a person without bushy eyebrows"],
    "Chubby":               ["a thin person", "a person with a slim face", "a person who is not chubby"],
    "Double_Chin":          ["a person without a double chin", "a person with a defined jawline"],
    "Eyeglasses":           ["a person without eyeglasses", "a person not wearing glasses", "a face without glasses", "a person with bare eyes"],
    "Goatee":               ["a person without a goatee", "a clean-shaven person", "a person with no goatee"],
    "Gray_Hair":            ["a person without gray hair", "a person whose hair is not gray"],
    "Heavy_Makeup":         ["a person with no makeup", "a person without makeup", "a face without heavy makeup", "a person with a bare natural face"],
    "High_Cheekbones":      ["a person without high cheekbones", "a person with flat cheeks"],
    "Male":                 ["a woman", "a female person", "a portrait of a woman", "a female face"],
    "Mouth_Slightly_Open":  ["a person with a closed mouth", "a person with closed lips", "a person whose mouth is shut"],
    "Mustache":             ["a clean-shaven person", "a person without a mustache", "a person with no mustache"],
    "Narrow_Eyes":          ["a person with wide eyes", "a person with big eyes", "a person without narrow eyes"],
    "No_Beard":             ["a person with a beard", "a bearded person", "a person with facial hair"],
    "Oval_Face":            ["a person without an oval face", "a person with a round face", "a person with a square face"],
    "Pale_Skin":            ["a person with dark skin", "a person with a tanned complexion", "a person without pale skin"],
    "Pointy_Nose":          ["a person with a rounded nose", "a person without a pointy nose"],
    "Receding_Hairline":    ["a person with a full hairline", "a person without a receding hairline"],
    "Rosy_Cheeks":          ["a person without rosy cheeks", "a person with pale cheeks"],
    "Sideburns":            ["a clean-shaven person", "a person without sideburns"],
    "Smiling":              ["a person with a neutral expression", "a person who is not smiling", "a serious-looking person", "a person with a straight face"],
    "Straight_Hair":        ["a person with curly hair", "a person without straight hair"],
    "Wavy_Hair":            ["a person with straight hair", "a person without wavy hair"],
    "Wearing_Earrings":     ["a person without earrings", "a person not wearing any earrings"],
    "Wearing_Hat":          ["a person without a hat", "a person not wearing a hat", "a bare-headed person"],
    "Wearing_Lipstick":     ["a person without lipstick", "a person with bare lips"],
    "Wearing_Necklace":     ["a person without a necklace", "a person with a bare neck"],
    "Wearing_Necktie":      ["a person without a necktie", "a person with an open collar"],
    "Young":                ["an old person", "an elderly person", "an older person", "a senior person"],
}

In [ ]:
# CLIP's official ImageNet 80-template set (the canonical zero-shot ensemble) plus a
# small number of portrait-specific templates appropriate to CelebA.
clip_imagenet_templates = [
    "a bad photo of {phrase}.", "a photo of many {phrase}.", "a sculpture of {phrase}.",
    "a photo of the hard to see {phrase}.", "a low resolution photo of {phrase}.",
    "a rendering of {phrase}.", "graffiti of {phrase}.", "a bad photo of {phrase}.",
    "a cropped photo of {phrase}.", "a tattoo of {phrase}.", "the embroidered {phrase}.",
    "a photo of a hard to see {phrase}.", "a bright photo of {phrase}.", "a photo of a clean {phrase}.",
    "a photo of a dirty {phrase}.", "a dark photo of {phrase}.", "a drawing of {phrase}.",
    "a photo of my {phrase}.", "the plastic {phrase}.", "a photo of the cool {phrase}.",
    "a close-up photo of {phrase}.", "a black and white photo of {phrase}.", "a painting of {phrase}.",
    "a painting of {phrase}.", "a pixelated photo of {phrase}.", "a sculpture of {phrase}.",
    "a bright photo of {phrase}.", "a cropped photo of {phrase}.", "a plastic {phrase}.",
    "a photo of the dirty {phrase}.", "a jpeg corrupted photo of {phrase}.",
    "a blurry photo of {phrase}.", "a photo of {phrase}.", "a good photo of {phrase}.",
    "a rendering of {phrase}.", "a {phrase} in a video game.", "a photo of one {phrase}.",
    "a doodle of {phrase}.", "a close-up photo of {phrase}.", "a photo of {phrase}.",
    "the origami {phrase}.", "a sketch of {phrase}.", "a doodle of {phrase}.",
    "a origami {phrase}.", "a low resolution photo of {phrase}.", "the toy {phrase}.",
    "a rendition of {phrase}.", "a photo of the clean {phrase}.", "a photo of a large {phrase}.",
    "a rendition of {phrase}.", "a photo of a nice {phrase}.", "a photo of a weird {phrase}.",
    "a blurry photo of {phrase}.", "a cartoon {phrase}.", "art of {phrase}.",
    "a sketch of {phrase}.", "a embroidered {phrase}.", "a pixelated photo of {phrase}.",
    "itap of {phrase}.", "a jpeg corrupted photo of {phrase}.", "a good photo of {phrase}.",
    "a plushie {phrase}.", "a photo of the nice {phrase}.", "a photo of the small {phrase}.",
    "a photo of the weird {phrase}.", "the cartoon {phrase}.", "art of {phrase}.",
    "a drawing of {phrase}.", "a photo of the large {phrase}.", "a black and white photo of {phrase}.",
    "the plushie {phrase}.", "a dark photo of {phrase}.", "itap of {phrase}.",
    "graffiti of {phrase}.", "a toy {phrase}.", "itap of {phrase}.",
    "a photo of a cool {phrase}.", "a photo of a small {phrase}.", "a tattoo of {phrase}.",
]
portrait_templates = [
    "a portrait of {phrase}.",
    "a portrait photograph of {phrase}.",
    "a closeup headshot of {phrase}.",
    "a candid photo of {phrase}.",
    "a studio portrait of {phrase}.",
    "a high-resolution headshot of {phrase}.",
    "a face photo of {phrase}.",
    "a photo showing the face of {phrase}.",
    "a frontal photo of {phrase}.",
    "a clear photo of {phrase}.",
]
prompt_templates_v2 = clip_imagenet_templates + portrait_templates


@torch.no_grad()
def _encode_phrases_through_templates(phrases: list[str], templates: list[str]) -> torch.Tensor:
    """Encode every (phrase x template) pair, L2-normalize each, mean-pool, re-normalize.

    Batched in a single processor/model call for speed.
    """
    prompts = [template.format(phrase=phrase) for phrase in phrases for template in templates]
    embs = encode_texts(prompts, model, processor, device)   # (P, D), per-row normalized
    mean_emb = embs.mean(dim=0)
    return mean_emb / mean_emb.norm()


@torch.no_grad()
def precompute_attribute_pos_neg_embeddings() -> tuple[torch.Tensor, torch.Tensor]:
    """Return (E_pos, E_neg), each (40, 512) and L2-normalized.

    E_pos[i] = ensemble over (positive phrases for attribute i) x (templates)
    E_neg[i] = ensemble over (negative phrases for attribute i) x (templates)
    """
    pos_embs, neg_embs = [], []
    for name in attr_names:
        pos_embs.append(_encode_phrases_through_templates(humanized_mappings_pos[name], prompt_templates_v2))
        neg_embs.append(_encode_phrases_through_templates(humanized_mappings_neg[name], prompt_templates_v2))
    E_pos = torch.stack(pos_embs, dim=0)
    E_neg = torch.stack(neg_embs, dim=0)
    return E_pos, E_neg


print("Precomputing pos/neg attribute embeddings with the expanded template bank (this may take a minute)...")
E_POS, E_NEG = precompute_attribute_pos_neg_embeddings()
E_POS = E_POS.to(embeddings.device)
E_NEG = E_NEG.to(embeddings.device)
print(f"E_POS: {tuple(E_POS.shape)},  E_NEG: {tuple(E_NEG.shape)}")


def evaluate_pos_neg_logit_composition(
    annotations: list[dict],
    embeddings: torch.Tensor,
    E_pos: torch.Tensor,
    E_neg: torch.Tensor,
) -> list[dict]:
    """Per-attribute logit composition using separate positive and negative text embeddings.

    For each candidate image and each query component:
        + attr  contributes  cos(img, E_pos[attr])
        - attr  contributes  cos(img, E_neg[attr])
    """
    # Precompute (N_imgs, 40) score matrices for both heads
    img_pos = embeddings @ E_pos.T   # (N, 40)
    img_neg = embeddings @ E_neg.T   # (N, 40)

    evaluation_results = []
    for i, annotation in enumerate(annotations):
        print(f"Evaluating query Q{i+1}: {annotation.get('query', '')}")
        # Build two mask vectors: where +attrs are 1, and where -attrs are 1
        pos_mask = torch.zeros(len(attr_names), device=embeddings.device)
        neg_mask = torch.zeros(len(attr_names), device=embeddings.device)
        for component in get_text_query(annotation).split(","):
            component = component.strip()
            if not component:
                continue
            sign_char, attr_name = component[0], component[1:].strip()
            j = attribute2idx[attr_name]
            if sign_char == "+":
                pos_mask[j] = 1.0
            elif sign_char == "-":
                neg_mask[j] = 1.0

        scores = (img_pos @ pos_mask) + (img_neg @ neg_mask)  # (N,)

        query_image_idxs = get_source_image_idxs(annotation)
        query_evaluation_results = {}
        for query_image_idx in query_image_idxs:
            _, topk_indices = torch.topk(scores, k=11)
            topk_indices = topk_indices[topk_indices != query_image_idx][:10]
            retrieved_indices = topk_indices.tolist()

            image_query_eval_metrics = {}
            for k in [1, 5, 10]:
                image_query_eval_metrics[k] = evaluate_retrieval(
                    retrieved_indices=retrieved_indices,
                    ground_truth_indices=get_ground_truth_indices(annotation, query_image_idx),
                    k=k,
                )
            query_evaluation_results[query_image_idx] = image_query_eval_metrics

        evaluation_results.append(query_evaluation_results)
    return evaluation_results

In [ ]:
evaluation_results_promptv2 = evaluate_pos_neg_logit_composition(
    annotations=annotations,
    embeddings=embeddings,
    E_pos=E_POS,
    E_neg=E_NEG,
)
average_results_per_query_promptv2 = [
    compute_query_average_results(query_result) for query_result in evaluation_results_promptv2
]
plot_metrics_across_k(
    average_results_per_query_promptv2,
    title="Prompt Ensembling v2 (CLIP-80 + linguistic negatives + logit composition) — Performance across K",
)

In [ ]:
plot_methods_comparison(
    {
        "Baseline":                  average_results_per_query_baseline,
        "Method 2 — Logit Compose":  average_results_per_query_logit,
        "Method 3 — Prompt Ens. v2": average_results_per_query_promptv2,
    },
    title="Training-Free Method Comparison — per-query Recall@K and Precision@K",
)

# Training-Based Method — CoOp

Up to now every method is training-free: the CLIP weights and the prompts are fixed. **CoOp** ([Context Optimization, Zhou et al. 2022](https://arxiv.org/abs/2109.01134)) keeps CLIP entirely frozen but replaces the *handcrafted prefix* in each prompt with a small set of **learnable context tokens** — typically `M = 16` vectors in the same embedding space as CLIP's word embeddings.

### How we frame it for CelebA's compositional queries
CelebA has 40 binary attributes — every image is annotated with a 40-dim 0/1 label vector — so the natural objective is **multi-label binary classification**. For every attribute `a` we build two prompts:
* **positive:** `[ctx_1] [ctx_2] ... [ctx_M] a person with {attr_a}.`
* **negative:** `[ctx_1] [ctx_2] ... [ctx_M] a person without {attr_a}.`

The same `M` learnable context vectors are **shared across all 40 attributes** and across the positive/negative prompts (the canonical "shared context" CoOp variant). With `M = 16`, that is only `M · 512 ≈ 8k` trainable parameters — small enough to fit comfortably on free Colab.

The per-attribute logit is
$$\ell_a(\mathbf{x}) = \cos\!\big(\mathbf{e}_{\text{img}}(\mathbf{x}),\, \mathbf{e}_+^{(a)}\big) - \cos\!\big(\mathbf{e}_{\text{img}}(\mathbf{x}),\, \mathbf{e}_-^{(a)}\big)$$
(scaled by CLIP's `logit_scale`), and we train with **binary cross-entropy** against the multi-label ground truth.

### Test-time retrieval
After training, `e_+^{(a)}` and `e_-^{(a)}` are 40 learned text embeddings. Retrieval is then identical to Methods 2 and 3:
$$
\text{score}(\mathbf{x}, q) = \sum_{i \in q^+} \cos(\mathbf{e}_{\text{img}}(\mathbf{x}), \mathbf{e}_+^{(i)})
                              + \sum_{i \in q^-} \cos(\mathbf{e}_{\text{img}}(\mathbf{x}), \mathbf{e}_-^{(i)})
$$

This means Methods 2, 3, and CoOp all share the same retrieval-time framework — they differ only in *how* the per-attribute text embeddings are produced (single prompt / ensemble of handcrafted prompts / learned-context prompts).

### Compute budget
* **Image encoder is frozen.** We pre-extract CLIP image features for every training image **once** and cache them to disk. Training afterwards never touches the image encoder; each step is a single text-encoder forward over 80 prompts plus a small BCE backward — fast on a T4 or any free-Colab GPU.
* **Two configs available below** — pick one via `COOP_CONFIG`:
  * `"colab"` — 16-shot subset (~640 images), 30 epochs, batch 64. Runs in a few minutes on Colab.
  * `"vm"` — full CelebA train split (~162k images), 10 epochs, batch 256, AMP. For final-evaluation runs on the dedicated VM.

### Step 1 — Load the CelebA training split and pre-extract image features

We load the **train** split of CelebA (the previous cells only loaded `test`) and pre-extract every training image's CLIP image embedding **once**, caching to Drive at `clip_cache/train_embeddings.pt`. After this cell, training never touches the CLIP image encoder again.

The torchvision Google-Drive download for CelebA is unreliable (the Drive quota frequently returns HTTP 429/404). If the standard `download=True` path fails, the recommended fallback is to use the same `celeba.zip` already staged on Drive for the test split — many such bundles ship `list_eval_partition.txt` and the `img_align_celeba/` folder, so they already contain the train images. Set `CELEBA_TRAIN_ROOT` below to wherever your CelebA root lives.

In [ ]:
# ----- CoOp config -----
# "colab" = 16-shot subset, fast iteration. "vm" = full train split, final evaluation.
COOP_CONFIG = "colab"  # change to "vm" on the dedicated GPU

CELEBA_TRAIN_ROOT = Path("/content/datasets")
TRAIN_FEATS_PATH = Path(EVALUATION_CACHE_DIR) / "train_embeddings.pt"

from torch.utils.data import DataLoader, TensorDataset, Subset

# Step 1a — load the train split
print(f"Loading CelebA train split from {CELEBA_TRAIN_ROOT} ...")
celeba_train = CelebA(root=CELEBA_TRAIN_ROOT, split="train", download=False)
print(f"CelebA train split size: {len(celeba_train)}")

# Step 1b — pre-extract image features once and cache (shared utility, see Cell 10)
train_features, train_labels = get_encoded_dataset(
    celeba_train, model, processor, device, str(TRAIN_FEATS_PATH), batch_size=128
)
print(f"train_features dtype: {train_features.dtype}, device: {train_features.device}")


### Step 2 — Define the CoOp prompt learner

This is the only `nn.Module` with trainable parameters in the entire notebook. The CLIP image and text encoders are kept frozen; only the `M × 512` context matrix is updated by SGD.

**Implementation detail.** We construct each prompt's token IDs once (with `M` placeholder `"X"` tokens in the prefix), embed them via CLIP's token-embedding lookup, then in the forward pass we **overwrite** the `M` prefix-position embeddings with our learnable `ctx` parameter before running the text transformer manually. We pool at the EOS position (CLIP's text-feature convention) and project to the joint embedding space.

In [ ]:
# Note: CoOp does NOT route through encode_texts(). It operates one level below
# the encoder API — it overwrites token embeddings at positions [1:1+M] with the
# learnable `self.ctx` parameter before running the text transformer, which is
# why it tokenizes directly via `processor.tokenizer` and reimplements the
# encoder forward in `_encode_text_with_ctx`.

import torch.nn as nn
import torch.nn.functional as F


class CoOpPromptLearner(nn.Module):
    """Shared-context CoOp on top of HuggingFace CLIP.

    For each attribute we build a positive and a negative prompt of the form
        [SOS] [ctx_1] ... [ctx_M] a person with/without {attr}. [EOS] [PAD] ...
    The token-embedding lookup happens once at construction time; the forward
    pass replaces positions [1:1+M] with `self.ctx` before running the (frozen)
    text transformer.
    """

    def __init__(self, clip_model: CLIPModel, processor: CLIPProcessor, attr_names: list[str], M: int = 16):
        super().__init__()
        self.clip = clip_model
        self.M = M
        self.attr_names = attr_names
        self.num_attrs = len(attr_names)

        token_emb_layer = clip_model.text_model.embeddings.token_embedding
        embedding_dim = token_emb_layer.embedding_dim

        # M learnable context vectors, initialized with the standard CoOp Gaussian.
        ctx = torch.empty(M, embedding_dim)
        nn.init.normal_(ctx, std=0.02)
        self.ctx = nn.Parameter(ctx)

        tokenizer = processor.tokenizer
        max_len = clip_model.config.text_config.max_position_embeddings   # 77 for ViT-B/32
        placeholder = " ".join(["X"] * M)

        readable = [name.replace("_", " ").lower() for name in attr_names]
        pos_prompts = [f"{placeholder} a person with {name}." for name in readable]
        neg_prompts = [f"{placeholder} a person without {name}." for name in readable]

        pos = tokenizer(pos_prompts, padding="max_length", max_length=max_len, truncation=True, return_tensors="pt")
        neg = tokenizer(neg_prompts, padding="max_length", max_length=max_len, truncation=True, return_tensors="pt")
        self.register_buffer("pos_ids", pos["input_ids"])    # (40, 77)
        self.register_buffer("neg_ids", neg["input_ids"])    # (40, 77)

        # Freeze CLIP
        for p in self.clip.parameters():
            p.requires_grad_(False)

    def _encode_text_with_ctx(self, input_ids: torch.Tensor) -> torch.Tensor:
        """Return L2-normalized text features (N, D) for a batch of prompt token-id rows."""
        text_model = self.clip.text_model
        device_ = self.ctx.device
        input_ids = input_ids.to(device_)
        bsz, seq_len = input_ids.shape

        # Token embeddings (frozen CLIP) with the prefix overwritten by self.ctx
        token_emb = text_model.embeddings.token_embedding(input_ids)            # (B, L, D)
        token_emb = torch.cat(
            [
                token_emb[:, :1, :],                                            # [SOS]
                self.ctx.unsqueeze(0).expand(bsz, -1, -1),                      # learnable ctx
                token_emb[:, 1 + self.M:, :],                                   # attr tokens + EOS + PAD
            ],
            dim=1,
        )

        position_ids = torch.arange(seq_len, device=device_).unsqueeze(0)
        position_emb = text_model.embeddings.position_embedding(position_ids)   # (1, L, D)
        hidden_states = token_emb + position_emb                                # (B, L, D)

        # Causal mask (upper-triangular -inf, zeros below diagonal)
        causal_mask = torch.full((seq_len, seq_len), float("-inf"), device=device_)
        causal_mask = torch.triu(causal_mask, diagonal=1)
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0).expand(bsz, 1, seq_len, seq_len)

        encoder_outputs = text_model.encoder(
            inputs_embeds=hidden_states,
            attention_mask=None,
            causal_attention_mask=causal_mask,
            output_attentions=False,
            output_hidden_states=False,
            return_dict=True,
        )
        last_hidden = text_model.final_layer_norm(encoder_outputs.last_hidden_state)

        # CLIP convention: pool at the EOS token (highest id in the sequence)
        eos_pos = input_ids.argmax(dim=-1)                                       # (B,)
        pooled = last_hidden[torch.arange(bsz, device=device_), eos_pos]
        text_features = self.clip.text_projection(pooled)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)
        return text_features

    def forward(self):
        e_pos = self._encode_text_with_ctx(self.pos_ids)   # (40, 512)
        e_neg = self._encode_text_with_ctx(self.neg_ids)   # (40, 512)
        return e_pos, e_neg


# Sanity check — instantiate and run a single forward
coop = CoOpPromptLearner(model, processor, list(celeba.attr_names), M=16).to(device)
with torch.no_grad():
    e_pos, e_neg = coop()
print(f"e_pos: {tuple(e_pos.shape)}, e_neg: {tuple(e_neg.shape)}")
n_trainable = sum(p.numel() for p in coop.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_trainable} (should be ~{16*512})")

### Step 3 — Training loop

We treat each of the 40 binary attributes as an independent binary classifier whose logit is `cos(img, e+) − cos(img, e−)` scaled by CLIP's `logit_scale`. The loss is **binary cross-entropy with logits** over the multi-label target.

We hold out 5% of the training set as a validation set and track macro-AUC across epochs to pick the best context.

In [ ]:
import math

# ----- Build train / val splits -----
torch.manual_seed(42)
np.random.seed(42)

# Make sure labels live on the GPU before we slice them.
train_labels_dev = train_labels.to(device) if isinstance(train_labels, torch.Tensor) else torch.as_tensor(train_labels, device=device)

if COOP_CONFIG == "colab":
    # 16-shot per attribute (positive AND negative). Per-attribute scan runs on GPU; the
    # `chosen` boolean mask handles deduplication without any Python set.
    K_SHOT = 16
    generator = torch.Generator(device=device).manual_seed(42)
    chosen = torch.zeros(train_labels_dev.shape[0], dtype=torch.bool, device=device)
    for j in range(train_labels_dev.shape[1]):
        for value in (1, 0):
            idxs = (train_labels_dev[:, j] == value).nonzero(as_tuple=True)[0]
            if idxs.numel() == 0:
                continue
            k = min(K_SHOT, int(idxs.numel()))
            pick = idxs[torch.randperm(idxs.numel(), generator=generator, device=device)[:k]]
            chosen[pick] = True
    selected = chosen.nonzero(as_tuple=True)[0]
    train_feats_sel = train_features[selected]
    train_labels_sel = train_labels_dev[selected]
    EPOCHS, BATCH, LR = 30, 64, 2e-3
    USE_AMP = False
elif COOP_CONFIG == "vm":
    train_feats_sel = train_features
    train_labels_sel = train_labels_dev
    EPOCHS, BATCH, LR = 10, 256, 2e-3
    USE_AMP = (device == "cuda")
else:
    raise ValueError(f"Unknown COOP_CONFIG: {COOP_CONFIG}")

# 5% validation split — indices live on GPU so the slices below never touch CPU.
N = train_feats_sel.shape[0]
perm = torch.randperm(N, device=device)
val_size = max(1, int(round(0.05 * N)))
val_idx = perm[:val_size]
trn_idx = perm[val_size:]
X_tr, Y_tr = train_feats_sel[trn_idx], train_labels_sel[trn_idx].float()
X_va, Y_va = train_feats_sel[val_idx], train_labels_sel[val_idx].float()

print(f"CoOp config: {COOP_CONFIG}  |  train: {tuple(X_tr.shape)}  val: {tuple(X_va.shape)}")
print(f"epochs={EPOCHS} batch={BATCH} lr={LR} M=16 amp={USE_AMP}")

# ----- Build a fresh learner and optimizer -----
coop = CoOpPromptLearner(model, processor, list(celeba.attr_names), M=16).to(device)
optimizer = torch.optim.SGD([coop.ctx], lr=LR, momentum=0.9)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
logit_scale = model.logit_scale.exp().detach()   # frozen, taken from pretrained CLIP

# Track best validation macro-AUC — GPU-resident Mann-Whitney AUC, no NumPy detour.
@torch.no_grad()
def macro_auc(logits: torch.Tensor, targets: torch.Tensor) -> float:
    """Macro-averaged AUC over the 40 attributes, computed on `logits.device`."""
    logits = logits.detach()
    targets = targets.detach()
    aucs: list[torch.Tensor] = []
    n = targets.shape[0]
    for j in range(targets.shape[1]):
        y = targets[:, j]
        n_pos = int(y.sum().item())
        if n_pos == 0 or n_pos == n:
            continue
        n_neg = n - n_pos
        order = torch.argsort(logits[:, j])
        ranks = torch.empty_like(order, dtype=torch.float32)
        ranks[order] = torch.arange(n, device=order.device, dtype=torch.float32)
        pos_ranks = ranks[y == 1].sum()
        aucs.append((pos_ranks - n_pos * (n_pos - 1) / 2) / (n_pos * n_neg))
    if not aucs:
        return float("nan")
    return float(torch.stack(aucs).mean())

best_val = -1.0
best_ctx = coop.ctx.detach().clone()
n_train = X_tr.shape[0]

print("Starting CoOp training...")
for epoch in range(EPOCHS):
    coop.train()
    perm = torch.randperm(n_train, device=device)
    total_loss = 0.0
    for start in range(0, n_train, BATCH):
        idx = perm[start:start + BATCH]
        x = X_tr[idx]       # (B, 512), already L2-normalized
        y = Y_tr[idx]       # (B, 40), float
        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            e_pos, e_neg = coop()                                      # (40, 512)
            logits = logit_scale * (x @ e_pos.T - x @ e_neg.T)         # (B, 40)
            loss = F.binary_cross_entropy_with_logits(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += float(loss.detach()) * x.shape[0]

    scheduler.step()

    # Validation
    coop.eval()
    with torch.no_grad():
        e_pos, e_neg = coop()
        val_logits = logit_scale * (X_va @ e_pos.T - X_va @ e_neg.T)
        val_auc = macro_auc(val_logits, Y_va)

    avg_loss = total_loss / max(1, n_train)
    print(f"Epoch {epoch+1:3d}/{EPOCHS}  loss={avg_loss:.4f}  val_macroAUC={val_auc:.4f}")

    if val_auc > best_val:
        best_val = val_auc
        best_ctx = coop.ctx.detach().clone()

# Reload the best context for retrieval
coop.ctx.data.copy_(best_ctx)
print(f"Best validation macro-AUC: {best_val:.4f}")

# Persist context to disk so retrieval can be re-run without retraining.
# `.cpu()` is intentional only at save time — the in-memory ctx stays on GPU.
COOP_CKPT = Path(cache_dir) / f"coop_ctx_{COOP_CONFIG}.pt"
torch.save({"ctx": best_ctx.cpu(), "M": 16, "config": COOP_CONFIG, "val_macro_auc": best_val}, COOP_CKPT)
print(f"Saved context to {COOP_CKPT}")


### Step 4 — Retrieval evaluation with the learned context

We freeze CLIP again and re-use the same per-attribute logit-composition framework as Method 2/3, this time with the learned `E_POS` and `E_NEG` produced by the trained CoOp learner.

In [ ]:
coop.eval()
with torch.no_grad():
    E_POS_COOP, E_NEG_COOP = coop()
E_POS_COOP = E_POS_COOP.to(embeddings.device)
E_NEG_COOP = E_NEG_COOP.to(embeddings.device)

evaluation_results_coop = evaluate_pos_neg_logit_composition(
    annotations=annotations,
    embeddings=embeddings,
    E_pos=E_POS_COOP,
    E_neg=E_NEG_COOP,
)
average_results_per_query_coop = [
    compute_query_average_results(query_result) for query_result in evaluation_results_coop
]
plot_metrics_across_k(
    average_results_per_query_coop,
    title=f"CoOp (config={COOP_CONFIG}, M=16) — Performance across K",
)

## Final Comparison — Baseline vs. Method 2 vs. Method 3 vs. CoOp

All four methods evaluated on the same benchmark JSON and the same precomputed image embeddings.

In [ ]:
plot_methods_comparison(
    {
        "Baseline":                  average_results_per_query_baseline,
        "Method 2 — Logit Compose":  average_results_per_query_logit,
        "Method 3 — Prompt Ens. v2": average_results_per_query_promptv2,
        f"CoOp ({COOP_CONFIG}, M=16)": average_results_per_query_coop,
    },
    title="Final Method Comparison — per-query Recall@K and Precision@K",
)